# Smartphone activity recognition: reproducible walkthrough

Run from the project folder with the pinned environment. The cells below have been executed against the supplied results. To rerun training, use `python -m src.experiment --output results_reproduction` in a terminal. The paper evaluates missing **derived features**, not physical sensor failures.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
from src.data import load_split
from src.models import predict
X, y, subjects = load_split('train')
Xt, yt, test_subjects = load_split('test')
print('Training:', X.shape, 'subjects:', len(np.unique(subjects)))
print('Test:', Xt.shape, 'subjects:', len(np.unique(test_subjects)))
print('Subject overlap:', set(subjects) & set(test_subjects))

Training: (7352, 561) subjects: 21
Test: (2947, 561) subjects: 9
Subject overlap: set()


## Inspect selection
Configurations were chosen using the training subjects only. The primary baseline was selected before loading test arrays.

In [2]:
selected = json.loads(Path('results/selection.json').read_text())
for name in ['LR','RBF-SVM','Random forest','Dropout-LR']:
    print(name, selected[name])
print('Primary comparator:', selected['primary_baseline'])

LR {'params': {'family': 'lr', 'C': 0.1}, 'cv_score': 0.8938311915540744}
RBF-SVM {'params': {'family': 'svm', 'C': 1.0}, 'cv_score': 0.8837538834914084}
Random forest {'params': {'family': 'rf', 'max_features': 'sqrt', 'trees': 200}, 'cv_score': 0.8709121685726798}
Dropout-LR {'params': {'family': 'dropout_lr', 'C': 0.1, 'p': 0.1}, 'cv_score': 0.8936488505513817}
Primary comparator: LR


## Inspect measured results
Macro-F1 is averaged over three masks in corrupted conditions. Clean scores use one unmodified test set.

In [3]:
scores = pd.read_csv('results/test_metrics.csv')
main = ['LR','RBF-SVM','Random forest','Dropout-LR']
clean = scores[(scores.kind=='clean') & scores.model.isin(main)]
print(clean[['model','accuracy','macro_f1']].to_string(index=False))
missing = scores[(scores.kind=='dropout') & (scores.severity==0.5) & scores.model.isin(main)]
print('\n50% missing entries:')
print(missing.groupby('model').macro_f1.mean().to_string())

        model  accuracy  macro_f1
           LR  0.949779  0.950026
   Dropout-LR  0.948761  0.949034
      RBF-SVM  0.952155  0.951479
Random forest  0.926366  0.924462

50% missing entries:
model
Dropout-LR       0.871057
LR               0.860736
RBF-SVM          0.789268
Random forest    0.752758


## Paired uncertainty
The bootstrap resamples subjects, preserving each subject's complete set of windows. It is conditional on fixed training and corruption draws.

In [4]:
comparisons = pd.read_csv('results/paired_comparisons.csv')
print(comparisons[comparisons.reference=='LR'].to_string(index=False))

   kind  severity reference     delta       low     high
  clean       0.0        LR -0.000992 -0.003888 0.001684
dropout       0.1        LR  0.000902 -0.002733 0.003918
dropout       0.3        LR  0.007199  0.005064 0.009544
dropout       0.5        LR  0.010320  0.006388 0.014433
dropout       0.7        LR  0.012803  0.007162 0.017621
  noise       0.1        LR -0.000893 -0.003439 0.001989
  noise       0.3        LR -0.000068 -0.003004 0.002688
  noise       0.5        LR  0.000841 -0.002784 0.004073
  noise       1.0        LR  0.004977 -0.001638 0.010901


## Make predictions
The model expects the original 561-feature order. Unavailable features can be `NaN` and are imputed with training means.

In [5]:
model = joblib.load('results/models/dropout-lr.joblib')
example = np.loadtxt('examples/features.csv',delimiter=',')
missing_example = np.loadtxt('examples/features_missing.csv',delimiter=',')
print('Clean example predictions:', predict(model,example))
print('Missing-feature predictions:', predict(model,missing_example))
print('Ground truth:', pd.read_csv('examples/ground_truth.csv').true_activity_id.to_numpy())

Clean example predictions: [1 2 3 4 5 6]
Missing-feature predictions: [1 2 3 4 5 6]
Ground truth: [1 2 3 4 5 6]


## Read the figures and paper

![Feature-loss curve](figures/dropout_robustness.png)

The complete paper is `paper/research_paper.pdf`; its editable text is `paper/research_paper.md`. See `paper/RESEARCH_GUIDE.md` for concepts, limitations and next research steps.